# Chapter 7 · 센서 누수·경보 정확도

**연구질문:** 센서 특성의 누수가 경보 정확도를 어떻게 부풀리며, 누수 없는 경보의 정밀도·재현율·선행시간은?

> 위에서 아래로 실행하면 끝까지 동작합니다.

## 1. 핵심 개념·수식

**누수(leakage):** 예측 시점 $t$에 알 수 없는 정보가 특성·전처리·분할에 들어가 성능이 과대평가되는 문제. 대표 경로 세 가지.
1. **전처리 누수:** 표준화·결측대체·모수 추정을 전체 자료로 한다(학습구간 통계만 써야 함).
2. **미래 정보 누수:** 중심 이동평균 등 $y_{t+k}$를 포함하는 특성.
3. **분할 누수:** 시계열에 무작위 K-겹 교차검증을 쓴다(인접 관측이 학습·검증에 섞임).

**Walk-forward(롤링 오리진) 평가:** 시점 $T_1<T_2<\dots$마다 $[1,T_k]$로 학습하고 $(T_k,T_k+h]$를 평가한다. 폴드별 오차의 분포(평균·분산·최악)를 본다.

**Diebold–Mariano 검정:** 손실차 $d_t=L(e^{A}_t)-L(e^{B}_t)$의 평균이 0인지 검정한다. $\mathrm{DM}=\bar d/\sqrt{\widehat{\mathrm{LRV}}(d)/n}$.

**부트스트랩 신뢰구간:** 폴드 손실차를 블록 부트스트랩하여 우열의 불확실성을 본다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def synthetic_sensor(n_days=180, seed=SEED, n_faults=14):
    # NOT real data: 일주기 + 느린 드리프트 + 잡음, 고장 직전 12시간 동안 진동이 증가하는 전조 구간 포함
    rng = np.random.default_rng(seed); n = n_days * 24; t = np.arange(n)
    vib = 1.0 + 0.15 * np.sin(2 * np.pi * (t % 24) / 24) + 0.0005 * t + rng.normal(0, 0.05, n)
    temp = 60 + 3 * np.sin(2 * np.pi * (t % 24) / 24 - 0.5) + rng.normal(0, 0.6, n)
    fault = np.zeros(n, int)
    starts = np.sort(rng.choice(np.arange(72, n - 24), n_faults, replace=False))
    for s0 in starts:
        for k in range(12):
            vib[s0 - 12 + k] += 0.03 * (k + 1); temp[s0 - 12 + k] += 0.15 * (k + 1)
        vib[s0:s0 + 2] += 0.8; fault[s0] = 1
    idx = pd.date_range("2024-01-01", periods=n, freq="h")
    return pd.DataFrame({"vibration": vib, "temp": temp, "fault": fault}, index=idx)

def synthetic_demand(n=1095, seed=SEED + 2):
    rng = np.random.default_rng(seed); t = np.arange(n)
    mu = 500 + 0.15 * t + 40 * np.sin(2 * np.pi * t / 365.25) + 60 * (t % 7 < 5) + rng.normal(0, 12, n)
    return pd.Series(mu, index=pd.date_range("2021-01-01", periods=n, freq="D"), name="demand")

def load_manufacturing(which="sensor"):
    # 달력 시간이 있는 무료 제조 시계열은 드물다(카탈로그 3절). 기본은 합성 자료이며 NOT real data로 표시한다.
    if which == "demand":
        return dict(y=synthetic_demand(), m=7, freq="D", label="일별 제조 수요(합성)", unit="개",
                    kind="SYNTHETIC (NOT real data)")
    df = synthetic_sensor()
    return dict(y=df["vibration"], df=df, m=24, freq="h", label="진동 센서(합성)", unit="mm/s",
                kind="SYNTHETIC (NOT real data)")

D = load_manufacturing("sensor")
y, DATA_KIND, M, FREQ = D["y"], D["kind"], D["m"], D["freq"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min(), "~", y.index.max())

## 2. 누수 실험

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve
df_m = D["df"].copy(); HOR = 12
fut = pd.concat([df_m["fault"].shift(-k) for k in range(1, HOR + 1)], axis=1).max(axis=1)
target = fut.fillna(0).astype(int)
def feats(s, name, center=False):
    f = pd.DataFrame(index=s.index)
    for w in (6, 12, 24):
        f[f"{name}_mean{w}"] = s.rolling(w, center=center).mean()           # center=True면 미래 값이 섞인다(누수)
        f[f"{name}_std{w}"] = s.rolling(w, center=center).std()
    return f
def build(center):
    X = pd.concat([feats(df_m["vibration"], "vib", center), feats(df_m["temp"], "temp", center)], axis=1)
    d = pd.concat([X, target.rename("t")], axis=1).dropna(); return d.drop(columns="t"), d["t"]
def split_eval(center, shuffle=False):
    X, t = build(center); n_tr = int(len(X) * 0.7)
    if shuffle:                                                             # 누수: 시간을 섞는 무작위 분할
        idx = np.random.default_rng(SEED).permutation(len(X)); tr_i, te_i = idx[:n_tr], idx[n_tr:]
    else:
        tr_i, te_i = np.arange(n_tr), np.arange(n_tr, len(X))
    m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, random_state=SEED).fit(X.iloc[tr_i], t.iloc[tr_i])
    p = m.predict_proba(X.iloc[te_i])[:, 1]; return average_precision_score(t.iloc[te_i], p), t.iloc[te_i].mean(), p, t.iloc[te_i], X.index[te_i]
rows = []
for label, c_, sh_ in [("정상: 과거 창 + 시간순 분할", False, False), ("누수A: 중심 창(미래 포함)", True, False),
                       ("누수B: 무작위 분할", False, True), ("누수A+B", True, True)]:
    ap, base_rate, *_ = split_eval(c_, sh_); rows.append({"설정": label, "PR_AUC": ap, "양성비율(기준)": base_rate})
show(pd.DataFrame(rows).set_index("설정"))
print("PR-AUC가 정상 설정보다 크게 높은 설정은 누수로 부풀려진 값이다(참 성능이 아님).")

## 3. 경보 품질

In [ ]:
# ---- 정상 설정에서 경보 품질: 임계값은 학습구간에서만 정하고 선행시간을 계산 ----
X, t = build(False); n_tr = int(len(X) * 0.7)
m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, random_state=SEED).fit(X.iloc[:n_tr], t.iloc[:n_tr])
p_tr = m.predict_proba(X.iloc[:n_tr])[:, 1]; p_te = m.predict_proba(X.iloc[n_tr:])[:, 1]
prec, rec, thr = precision_recall_curve(t.iloc[:n_tr], p_tr); ok_ = np.where(prec[:-1] >= 0.5)[0]
th = float(thr[ok_[0]]) if len(ok_) else 0.5                         # 학습구간에서 정밀도>=0.5가 되는 최소 임계값
alarm = pd.Series(p_te >= th, index=X.index[n_tr:])
events = df_m.index[(df_m["fault"] == 1) & (df_m.index >= X.index[n_tr])]
lead, caught = [], 0
for ev in events:                                                      # 이벤트 전 12시간 안에 첫 경보까지의 선행시간
    win = alarm.loc[ev - pd.Timedelta(hours=HOR): ev - pd.Timedelta(hours=1)]
    if win.any():
        caught += 1; lead.append((ev - win[win].index[0]).total_seconds() / 3600)
false_alarm_hours = int((alarm & ~(t.loc[alarm.index] == 1)).sum())
print(f"검증구간 고장 {len(events)}건 중 사전 경보 {caught}건 | 평균 선행시간 {np.mean(lead) if lead else float('nan'):.1f}시간 | 오경보 시간 수 {false_alarm_hours} / {len(alarm)}")
print("주의: 이벤트가 매우 적어(수 건) 재현율·선행시간의 불확실성이 크다. 신뢰구간 또는 반복 검증이 필요하다.")

## 3. 결과 해석 작성
1. 정상 설정과 누수 설정의 성능 차이를 표로 제시하고, **어떤 누수 경로가 가장 큰 왜곡을 만들었는지** 설명한다.
2. 폴드(또는 지역·기간) 간 변동을 보고 "평균은 좋지만 최악의 경우가 나쁘다"를 구분한다.
3. 결론(모형 우열, 효과 방향)이 설정 변경에도 유지되는가? 유지되지 않으면 "강건하다"가 아니라 "민감하다"로 쓴다.
4. 한계: 폴드 수, 이벤트 수, 합성자료의 한계, 위약검정이 모든 식별 가정을 증명하지는 않는다는 점.

**제출:** 정상/누수 비교표, 폴드별 결과, 강건성 표, 한계 단락. 누수 점검 목록(`validation/Leakage_Checklist.md`)과 대조하라.